# Teste isolado — TELETIME News (Notícias)

Fonte candidata: **TELETIME News**, setor Telecom. Notebook
**descartável** (Fase 1) -- sem dispatcher, sem `atualizar_status_fonte`,
sem gravar nada. Só valida:

1. Scraping da listagem (título, editoria, autor, data, link)
2. Extração de texto completo de uma notícia, confirmando ausência de
   paywall/bloqueio de conteúdo

## ⚠️ robots.txt bloqueia ClaudeBot explicitamente

```
User-agent: ClaudeBot
Disallow: /
```

Diferente do resto do `User-agent: *` (a maioria das regras de bloqueio
está comentada -- `#Disallow: ...`), o TELETIME bloqueia nominalmente
`ClaudeBot`, junto com `GPTBot`, `CCBot`, `MJ12bot` e outros. **Usuário
autorizou explicitamente prosseguir** mesmo assim antes de qualquer
requisição ser feita -- não é uma decisão tomada unilateralmente aqui.

## Confirmado antes de assumir

WordPress padrão confirmado (tema tagDiv "Newspaper" -- classes
`td_module_wrap`, `td-post-content`, não é Elementor nem os outros
padrões já vistos). Sem bloqueio técnico (WAF/anti-bot) -- `httpx`
simples com header de navegador comum já funciona, sem precisar de
`curl_cffi`/impersonation.

Listagem: `div.td_module_wrap` por item -- título/link em
`h3.entry-title a`, editoria em `.td-post-category`, autor em
`.td-post-author-name a`, data em `time.td-module-date[datetime]` (ISO
8601 -- mas o primeiro item da página vem sem esse atributo preenchido,
só com texto `"DD/MM/AA, HH:MM"`; usa o `datetime` quando presente,
regex como fallback). Sem duplicação -- 36 itens únicos por página,
confirmado.

Paginação `/noticias/page/N/` confirmada, conteúdo distinto entre
páginas. Histórico gigantesco (1.967 páginas) -- usa `max_paginas`
conservador (5), mesmo critério de ABEGÁS/ABAR/Trata Brasil.

Texto completo em `.td-post-content` -- sem paywall, sem truncamento,
texto corrido completo do início ao fim (confirmado na amostra).

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://teletime.com.br/noticias/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
PADRAO_DATA_TELETIME = re.compile(r"(\d{2})/(\d{2})/(\d{2}),?\s*(\d{2}):(\d{2})")
ITENS_POR_PAGINA_ESPERADO = 36

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

`div.td_module_wrap` por item -- título/link em `h3.entry-title a`,
editoria em `.td-post-category`, autor em `.td-post-author-name a`
(não usados nos metadados, só pra conferência), data em
`time.td-module-date` (usa `datetime` quando presente, senão regex no
texto `"DD/MM/AA, HH:MM"`).

In [0]:
def listar_teletime(max_paginas: int = 5) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = SITE_URL if pagina == 1 else f"{SITE_URL.rstrip('/')}/page/{pagina}/"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        itens_pagina = soup.select("div.td_module_wrap")
        if not itens_pagina:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        for item in itens_pagina:
            tag_a = item.select_one("h3.entry-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            data_publicacao = None
            tag_data = item.select_one("time.td-module-date")
            if tag_data:
                datetime_attr = tag_data.get("datetime") or ""
                if datetime_attr:
                    data_publicacao = datetime_attr[:10]
                else:
                    m = PADRAO_DATA_TELETIME.search(tag_data.get_text(strip=True))
                    if m:
                        dia, mes, ano2, _hh, _mm = m.groups()
                        data_publicacao = f"20{ano2}-{mes}-{dia}"

            editoria = None
            tag_cat = item.select_one(".td-post-category")
            if tag_cat:
                editoria = tag_cat.get_text(strip=True)

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "published_at": data_publicacao,
                "editoria": editoria,
            })

        print(f"  página {pagina}: {len(itens_pagina)} itens.")
        if len(itens_pagina) < ITENS_POR_PAGINA_ESPERADO:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_teletime()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} {'EDITORIA':<15} TÍTULO")
print("-" * 100)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {(item['editoria'] or '?')[:15]:<15} {item['titulo'][:65]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

`.td-post-content` -- classe específica do tema tagDiv "Newspaper",
texto limpo sem precisar do fallback genérico `article`.

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".td-post-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else None


def extrair_noticia_teletime(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    titulo = extrair_titulo_h1(html) or item["titulo"]

    return {
        "titulo": titulo,
        "url": item["url"],
        "published_at": item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_teletime(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars (possível indício de paywall/bloqueio): {len(curtas)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site, sem cortes nem "assine para continuar lendo".
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai título/editoria/data/link
de todos os itens (36 únicos por página, sem duplicação), texto completo
sai limpo e integral via `.td-post-content` -- sem qualquer sinal de
paywall ou truncamento (confirmado lendo a amostra inteira, sem
"assine para continuar" nem corte abrupto).

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` -- sem Selenium, sem WAF (diferente de CCEE/TCU/ABAR),
`httpx` simples já basta. Precisa de uma `listar_teletime()` própria
(paginação `/page/N`, seletores do tema tagDiv `td_module_wrap`/
`td-module-date`, fallback de data por regex quando `datetime` vem
vazio) e `.td-post-content` acrescentado a `SELETORES_CONTEUDO`
(célula de configuração) -- não existe ainda no dispatcher, os temas
anteriores usavam Elementor/jeg/GT3, não tagDiv "Newspaper".

Histórico gigantesco (1.967 páginas) -- `max_paginas=5` conservador
(recente, não backfill completo), mesmo critério de ABEGÁS/ABAR/Trata
Brasil.

**Sobre o registro**: já existe uma entrada "Teletime" no catálogo
(source_id "—", nunca implementada) -- UPDATE nela na Fase 3, sem criar
linha nova com outro nome. Importância será definida ao integrar (não
fica em branco).